<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Space X  Falcon 9 First Stage Landing Prediction**


## Web scraping Falcon 9 and Falcon Heavy Launches Records from Wikipedia


Estimated time needed: **40** minutes


In this lab, you will be performing web scraping to collect Falcon 9 historical launch records from a Wikipedia page titled `List of Falcon 9 and Falcon Heavy launches`

https://en.wikipedia.org/wiki/List_of_Falcon_9_and_Falcon_Heavy_launches


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/Falcon9_rocket_family.svg)


Falcon 9 first stage will land successfully


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/landing_1.gif)


Several examples of an unsuccessful landing are shown here:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBMDeveloperSkillsNetwork-DS0701EN-SkillsNetwork/api/Images/crash.gif)


More specifically, the launch records are stored in a HTML table shown below:


![](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_1_L2/images/falcon9-launches-wiki.png)


  ## Objectives
Web scrap Falcon 9 launch records with `BeautifulSoup`: 
- Extract a Falcon 9 launch records HTML table from Wikipedia
- Parse the table and convert it into a Pandas data frame


First let's import required packages for this lab


In [1]:
!pip3 install beautifulsoup4
!pip3 install requests


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import sys

import requests
from bs4 import BeautifulSoup
import re
import unicodedata
import pandas as pd

and we will provide some helper functions for you to process web scraped HTML table


In [3]:
def date_time(table_cells):
    """
    This function returns the data and time from the HTML  table cell
    Input: the  element of a table data cell extracts extra row
    """
    return [data_time.strip() for data_time in list(table_cells.strings)][0:2]

def booster_version(table_cells):
    """
    This function returns the booster version from the HTML  table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=''.join([booster_version for i,booster_version in enumerate( table_cells.strings) if i%2==0][0:-1])
    return out

def landing_status(table_cells):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    out=[i for i in table_cells.strings][0]
    return out


def get_mass(table_cells):
    mass=unicodedata.normalize("NFKD", table_cells.text).strip()
    if mass:
        mass.find("kg")
        new_mass=mass[0:mass.find("kg")+2]
    else:
        new_mass=0
    return new_mass


def extract_column_from_header(row):
    """
    This function returns the landing status from the HTML table cell 
    Input: the  element of a table data cell extracts extra row
    """
    if (row.br):
        row.br.extract()
    if row.a:
        row.a.extract()
    if row.sup:
        row.sup.extract()
        
    colunm_name = ' '.join(row.contents)
    
    # Filter the digit and empty names
    if not(colunm_name.strip().isdigit()):
        colunm_name = colunm_name.strip()
        return colunm_name    


To keep the lab tasks consistent, you will be asked to scrape the data from a snapshot of the  `List of Falcon 9 and Falcon Heavy launches` Wikipage updated on
`9th June 2021`


In [4]:
static_url = "https://en.wikipedia.org/w/index.php?title=List_of_Falcon_9_and_Falcon_Heavy_launches&oldid=1027686922"

headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                  "AppleWebKit/537.36 (KHTML, like Gecko) "
                  "Chrome/91.0.4472.124 Safari/537.36"
}

Next, request the HTML page from the above URL and get a `response` object


### TASK 1: Request the Falcon9 Launch Wiki page from its URL


First, let's perform an HTTP GET method to request the Falcon9 Launch HTML page, as an HTTP response.


In [8]:
# use requests.get() method with the provided static_url and headers
# assign the response to a object
# 1. Clear any stuck network states
response = requests.get(static_url, headers=headers)

Create a `BeautifulSoup` object from the HTML `response`


In [9]:
# Use BeautifulSoup() to create a BeautifulSoup object from a response text content
soup = BeautifulSoup(response.content, 'html.parser')

Print the page title to verify if the `BeautifulSoup` object was created properly 


In [10]:
# Use soup.title attribute
page_title = soup.title.string
print(page_title)

List of Falcon 9 and Falcon Heavy launches - Wikipedia


### TASK 2: Extract all column/variable names from the HTML table header


Next, we want to collect all relevant column names from the HTML table header


Let's try to find all tables on the wiki page first. If you need to refresh your memory about `BeautifulSoup`, please check the external reference link towards the end of this lab


In [12]:
# Use the find_all function in the BeautifulSoup object, with element type `table`
# Assign the result to a list called `html_tables`
# Initialize the launch_dict with the required keys as empty lists
launch_dict = {
    'Flight No.': [],
    'Launch site': [],
    'Payload': [],
    'Payload mass': [],
    'Orbit': [],
    'Customer': [],
    'Launch outcome': [],
    'Version Booster': [],
    'Booster landing': [],
    'Date': [],
    'Time': []
}

print("launch_dict initialized successfully.")

launch_dict initialized successfully.


Starting from the third table is our target table contains the actual launch records.


In [17]:
# Let's print the third table and check its content
extracted_row = 0
# 1. Broaden the search to any 'wikitable'
for table_number, table in enumerate(soup.find_all('table', "wikitable")):
    # Get table row 
    for rows in table.find_all("tr"):
        # Check if the first table heading is a number (Flight Number)
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
            else:
                flag = False
        else:
            flag = False
            
        # If it is a digit, it's a launch row
        if flag:
            extracted_row += 1
            cells = rows.find_all('td')
            
            # --- Your existing launch_dict.append calls go here ---
            # Make sure to include the logic from the previous cell inside this block!
            
            # Quick verification print every 10 rows
            if extracted_row % 20 == 0:
                print(f"Row {extracted_row} reached...")

print(f"Extraction complete! Total rows extracted: {extracted_row}")

Row 20 reached...
Row 40 reached...
Row 60 reached...
Row 80 reached...
Row 100 reached...
Row 120 reached...
Extraction complete! Total rows extracted: 121


In [22]:
import pandas as pd

# 1. Initialize the dictionary with empty lists
launch_dict = {
    'Flight No.': [], 'Launch site': [], 'Payload': [], 
    'Payload mass': [], 'Orbit': [], 'Customer': [], 
    'Launch outcome': [], 'Version Booster': [], 
    'Booster landing': [], 'Date': [], 'Time': []
}

extracted_row = 0
# 2. Extract from any table with the 'wikitable' class
for table_number, table in enumerate(soup.find_all('table', "wikitable")):
    for rows in table.find_all("tr"):
        if rows.th and rows.th.string:
            flight_number = rows.th.string.strip()
            if flight_number.isdigit():
                extracted_row += 1
                cells = rows.find_all('td')
                
                # --- START SAVING DATA ---
                launch_dict['Flight No.'].append(flight_number)
                
                # Date & Time
                datatimelist = date_time(cells[0])
                launch_dict['Date'].append(datatimelist[0].strip(','))
                launch_dict['Time'].append(datatimelist[1])
                
                # Booster Version
                bv = booster_version(cells[1])
                if not bv:
                    bv = cells[1].a.string if cells[1].a else "N/A"
                launch_dict['Version Booster'].append(bv)
                
                # Launch Site, Payload, Orbit (Standard text extraction)
                launch_dict['Launch site'].append(cells[2].get_text().strip())
                launch_dict['Payload'].append(cells[3].get_text().strip())
                launch_dict['Orbit'].append(cells[5].get_text().strip())
                
                # Payload Mass (Using the helper function)
                launch_dict['Payload mass'].append(get_mass(cells[4]))
                
                # Customer, Outcome, Landing (With safety checks)
                launch_dict['Customer'].append(cells[6].get_text().strip() if cells[6] else "N/A")
                launch_dict['Launch outcome'].append(list(cells[7].strings)[0] if cells[7] else "N/A")
                launch_dict['Booster landing'].append(landing_status(cells[8]))

# 3. Create and Display the DataFrame immediately
df = pd.DataFrame(launch_dict)
print(f"Extraction complete! {len(df)} rows saved.")
df.head(10)

Extraction complete! 121 rows saved.


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,"CCAFS,SLC-40",Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success\n,F9 v1.07B0003.18,Failure,4 June 2010,18:45
1,2,"CCAFS,SLC-40",Dragon demo flight C1(Dragon C101),0,LEO (ISS),NASA (COTS)\nNRO,Success,F9 v1.07B0004.18,Failure,8 December 2010,15:43
2,3,"CCAFS,SLC-40",Dragon demo flight C2+[18](Dragon C102),525 kg,LEO (ISS),NASA (COTS),Success,F9 v1.07B0005.18,No attempt\n,22 May 2012,07:44
3,4,"CCAFS,SLC-40",SpaceX CRS-1[22](Dragon C103),"4,700 kg",LEO (ISS),NASA (CRS),Success\n,F9 v1.07B0006.18,No attempt,8 October 2012,00:35
4,5,"CCAFS,SLC-40",SpaceX CRS-2[22](Dragon C104),"4,877 kg",LEO (ISS),NASA (CRS),Success\n,F9 v1.07B0007.18,No attempt\n,1 March 2013,15:10
5,6,"VAFB,SLC-4E",CASSIOPE[22][31],500 kg,Polar orbit LEO,MDA,Success,F9 v1.17B10038,Uncontrolled,29 September 2013,16:00
6,7,"CCAFS,SLC-40",SES-8[22][35][36],"3,170 kg",GTO,SES,Success,F9 v1.1,No attempt,3 December 2013,22:41
7,8,"CCAFS,SLC-40",Thaicom 6[22],"3,325 kg",GTO,Thaicom,Success,F9 v1.1,No attempt,6 January 2014,22:06
8,9,"Cape Canaveral,LC-40",SpaceX CRS-3[22](Dragon C105),"2,296 kg",LEO (ISS),NASA (CRS),Success\n,F9 v1.1,Controlled,18 April 2014,19:25
9,10,"Cape Canaveral,LC-40",Orbcomm-OG2-1(6 satellites)[22],"1,316 kg",LEO,Orbcomm,Success,F9 v1.1,Controlled,14 July 2014,15:15


In [25]:
import pandas as pd
from IPython.display import display, HTML

# 1. Create the DataFrame
df = pd.DataFrame(launch_dict)

# 2. This will print the ACTUAL <tr> and <td> tags in your console
print(df.head(5).to_html())

# 3. This will RENDER it as a formal boxed table
display(HTML(df.head(5).to_html()))

<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>Flight No.</th>
      <th>Launch site</th>
      <th>Payload</th>
      <th>Payload mass</th>
      <th>Orbit</th>
      <th>Customer</th>
      <th>Launch outcome</th>
      <th>Version Booster</th>
      <th>Booster landing</th>
      <th>Date</th>
      <th>Time</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>1</td>
      <td>CCAFS,SLC-40</td>
      <td>Dragon Spacecraft Qualification Unit</td>
      <td>0</td>
      <td>LEO</td>
      <td>SpaceX</td>
      <td>Success\n</td>
      <td>F9 v1.07B0003.18</td>
      <td>Failure</td>
      <td>4 June 2010</td>
      <td>18:45</td>
    </tr>
    <tr>
      <th>1</th>
      <td>2</td>
      <td>CCAFS,SLC-40</td>
      <td>Dragon demo flight C1(Dragon C101)</td>
      <td>0</td>
      <td>LEO (ISS)</td>
      <td>NASA (COTS)\nNRO</td>
      <td>Success</td>
      <td>F9 v1.07B0004.18</td>
      <td>Failure</t

,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,"CCAFS,SLC-40",Dragon Spacecraft Qualification Unit,0,LEO,SpaceX,Success\n,F9 v1.07B0003.18,Failure,4 June 2010,18:45
1,2,"CCAFS,SLC-40",Dragon demo flight C1(Dragon C101),0,LEO (ISS),NASA (COTS)\nNRO,Success,F9 v1.07B0004.18,Failure,8 December 2010,15:43
2,3,"CCAFS,SLC-40",Dragon demo flight C2+[18](Dragon C102),525 kg,LEO (ISS),NASA (COTS),Success,F9 v1.07B0005.18,No attempt\n,22 May 2012,07:44
3,4,"CCAFS,SLC-40",SpaceX CRS-1[22](Dragon C103),"4,700 kg",LEO (ISS),NASA (CRS),Success\n,F9 v1.07B0006.18,No attempt,8 October 2012,00:35
4,5,"CCAFS,SLC-40",SpaceX CRS-2[22](Dragon C104),"4,877 kg",LEO (ISS),NASA (CRS),Success\n,F9 v1.07B0007.18,No attempt\n,1 March 2013,15:10


You should able to see the columns names embedded in the table header elements `<th>` as follows:


```
<tr>
<th scope="col">Flight No.
</th>
<th scope="col">Date and<br/>time (<a href="/wiki/Coordinated_Universal_Time" title="Coordinated Universal Time">UTC</a>)
</th>
<th scope="col"><a href="/wiki/List_of_Falcon_9_first-stage_boosters" title="List of Falcon 9 first-stage boosters">Version,<br/>Booster</a> <sup class="reference" id="cite_ref-booster_11-0"><a href="#cite_note-booster-11">[b]</a></sup>
</th>
<th scope="col">Launch site
</th>
<th scope="col">Payload<sup class="reference" id="cite_ref-Dragon_12-0"><a href="#cite_note-Dragon-12">[c]</a></sup>
</th>
<th scope="col">Payload mass
</th>
<th scope="col">Orbit
</th>
<th scope="col">Customer
</th>
<th scope="col">Launch<br/>outcome
</th>
<th scope="col"><a href="/wiki/Falcon_9_first-stage_landing_tests" title="Falcon 9 first-stage landing tests">Booster<br/>landing</a>
</th></tr>
```


Next, we just need to iterate through the `<th>` elements and apply the provided `extract_column_from_header()` to extract column name one by one


In [38]:


# 1. First, we define the table (the 3rd 'wikitable' is usually the first launch table)
first_launch_table = soup.find_all('table', "wikitable")[2]

# 2. Next, we just need to iterate through the <th> elements 
# and apply the provided extract_column_from_header()
column_names = []

# Apply find_all() function with `th` element on first_launch_table
temp = first_launch_table.find_all('th')

# Iterate each th element and apply the provided extract_column_from_header() 
# to extract column name one by one
for column_name in temp:
    name = extract_column_from_header(column_name)
    
    # Append the Non-empty column name into a list called column_names
    if name is not None and len(name) > 0:
        column_names.append(name)


In [28]:
print(df['Version Booster'].unique())

<StringArray>
[  'F9 v1.07B0003.18',   'F9 v1.07B0004.18',   'F9 v1.07B0005.18',
   'F9 v1.07B0006.18',   'F9 v1.07B0007.18',     'F9 v1.17B10038',
            'F9 v1.1',           'F9 v1.1[',             'F9 FT[',
            'F9 FT♺[',    'F9 FTB1029.2195',             'F9 B4[',
    'F9 FTB1031.2220',    'F9 FTB1035.2227',    'F9 FTB1036.2227',
    'F9 FTB1032.2245',    'F9 FTB1038.2268',    'F9 B4B1041.2268',
    'F9 B4B1039.2292', 'F9 B5311B1046.1268',    'F9 B4B1043.2322',
    'F9 B4B1040.2268',    'F9 B4B1045.2336',              'F9 B5',
     'F9 B5349B1048[',    'F9 B5B1046.2354',             'F9 B5[',
    'F9 B5B1048.2364',    'F9 B5B1047.2268',    'F9 B5B1046.3268',
    'F9 B5B1049.2397',    'F9 B5B1048.3399',         'F9 B5[]413',
    'F9 B5B1049.3434',    'F9 B5B1051.2420',    'F9 B5B1056.2465',
    'F9 B5B1047.3472',    'F9 B5B1056.3482',    'F9 B5B1058.2544',
    'F9 B5B1049.6544',    'F9 B5B1060.2563',    'F9 B5B1058.3565',
    'F9 B5B1051.6568',           'F9 B5 ♺[',    

Check the extracted column names


In [39]:
print(column_names)

['Flight No.', 'Date and time ( )', 'Launch site', 'Payload', 'Payload mass', 'Orbit', 'Customer', 'Launch outcome']


## TASK 3: Create a data frame by parsing the launch HTML tables


We will create an empty dictionary with keys from the extracted column names in the previous task. Later, this dictionary will be converted into a Pandas dataframe


In [40]:
launch_dict= dict.fromkeys(column_names)

# Remove an irrelvant column
del launch_dict['Date and time ( )']

# Let's initial the launch_dict with each value to be an empty list
launch_dict['Flight No.'] = []
launch_dict['Launch site'] = []
launch_dict['Payload'] = []
launch_dict['Payload mass'] = []
launch_dict['Orbit'] = []
launch_dict['Customer'] = []
launch_dict['Launch outcome'] = []
# Added some new columns
launch_dict['Version Booster']=[]
launch_dict['Booster landing']=[]
launch_dict['Date']=[]
launch_dict['Time']=[]

Next, we just need to fill up the `launch_dict` with launch records extracted from table rows.


Usually, HTML tables in Wiki pages are likely to contain unexpected annotations and other types of noises, such as reference links `B0004.1[8]`, missing values `N/A [e]`, inconsistent formatting, etc.


To simplify the parsing process, we have provided an incomplete code snippet below to help you to fill up the `launch_dict`. Please complete the following code snippet with TODOs or you can choose to write your own logic to parse all launch tables:


In [42]:
extracted_row = 0
# Extract each table 
for table_number, table in enumerate(soup.find_all('table', "wikitable plainrowheaders collapsible")):
    # Get table row 
    for rows in table.find_all("tr"):
        # Check to see if first table heading is a number
        if rows.th:
            if rows.th.string:
                flight_number = rows.th.string.strip()
                flag = flight_number.isdigit()
        else:
            flag = False
            
        # Get table elements (td)
        row = rows.find_all('td')
        
        # If it is a number, save cells in the dictionary
        if flag:
            extracted_row += 1
            
            # 1. Flight Number
            launch_dict['Flight No.'].append(flight_number)
            
            # 2. Date & Time
            datatimelist = date_time(row[0])
            date = datatimelist[0].strip(',')
            launch_dict['Date'].append(date)
            
            time = datatimelist[1]
            launch_dict['Time'].append(time)
              
            # 3. Booster version
            bv = booster_version(row[1])
            if not(bv):
                bv = row[1].a.string if row[1].a else "N/A"
            launch_dict['Version Booster'].append(bv)
            
            # 4. Launch Site
            launch_site = row[2].a.string if row[2].a else row[2].get_text().strip()
            launch_dict['Launch site'].append(launch_site)
            
            # 5. Payload
            payload = row[3].a.string if row[3].a else row[3].get_text().strip()
            launch_dict['Payload'].append(payload)
            
            # 6. Payload Mass
            payload_mass = get_mass(row[4])
            launch_dict['Payload mass'].append(payload_mass)
            
            # 7. Orbit
            orbit = row[5].a.string if row[5].a else row[5].get_text().strip()
            launch_dict['Orbit'].append(orbit)
            
            # 8. Customer
            # Added a safety check here because row[6].a is often None for newer launches
            if row[6].a:
                customer = row[6].a.string
            else:
                customer = row[6].get_text().strip()
            launch_dict['Customer'].append(customer)
            
            # 9. Launch outcome
            launch_outcome = list(row[7].strings)[0] if row[7] else "N/A"
            launch_dict['Launch outcome'].append(launch_outcome)
            
            # 10. Booster landing
            booster_landing = landing_status(row[8])
            launch_dict['Booster landing'].append(booster_landing)

print(f"Extraction complete! Total rows extracted: {extracted_row}")
            

Extraction complete! Total rows extracted: 121


After you have fill in the parsed launch record values into `launch_dict`, you can create a dataframe from it.


In [43]:
df= pd.DataFrame({ key:pd.Series(value) for key, value in launch_dict.items() })

We can now export it to a <b>CSV</b> for the next section, but to make the answers consistent and in case you have difficulties finishing this lab. 

Following labs will be using a provided dataset to make each lab independent. 


In [45]:
df.to_csv('spacex_web_scraped.csv', index=False)


In [49]:
#To clean up the csv generation: 
import pandas as pd

# 1. Load the file
df = pd.read_csv('spacex_web_scraped.csv')

# 2. Clean 'Launch outcome' and 'Booster landing' (remove \n and extra spaces)
df['Launch outcome'] = df['Launch outcome'].str.replace('\n', '', regex=False).str.strip()
df['Booster landing'] = df['Booster landing'].str.replace('\n', '', regex=False).str.strip()

# 3. Clean 'Payload mass' (remove 'kg', commas, and convert to number)
df['Payload mass'] = df['Payload mass'].str.replace(' kg', '', regex=False)
df['Payload mass'] = df['Payload mass'].str.replace(',', '', regex=False)
df['Payload mass'] = pd.to_numeric(df['Payload mass'], errors='coerce')

# 4. Fill missing payload mass with the mean (as required by the lab)
df['Payload mass'].fillna(df['Payload mass'].mean(), inplace=True)

# 5. Save the final "Master" version
df.to_csv('spacex_web_scraped_cleaned.csv', index=False)

print("Final cleanup complete! Use 'spacex_web_scraped_cleaned.csv' for your presentation.")
df.head()

Final cleanup complete! Use 'spacex_web_scraped_cleaned.csv' for your presentation.


C:\Users\hp\AppData\Local\Temp\ipykernel_17452\1627085817.py:17: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment using an inplace method.
Such inplace method never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' instead, to perform the operation inplace on the original object, or try to avoid an inplace operation using 'df[col] = df[col].method(value)'.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html
  df['Payload mass'].fillna(df['Payload mass'].mean(), inplace=True)


,Flight No.,Launch site,Payload,Payload mass,Orbit,Customer,Launch outcome,Version Booster,Booster landing,Date,Time
0,1,CCAFS,Dragon Spacecraft Qualification Unit,0.0,LEO,SpaceX,Success,F9 v1.07B0003.18,Failure,4 June 2010,18:45
1,2,CCAFS,Dragon,0.0,LEO,NASA,Success,F9 v1.07B0004.18,Failure,8 December 2010,15:43
2,3,CCAFS,Dragon,525.0,LEO,NASA,Success,F9 v1.07B0005.18,No attempt,22 May 2012,07:44
3,4,CCAFS,SpaceX CRS-1,4700.0,LEO,NASA,Success,F9 v1.07B0006.18,No attempt,8 October 2012,00:35
4,5,CCAFS,SpaceX CRS-2,4877.0,LEO,NASA,Success,F9 v1.07B0007.18,No attempt,1 March 2013,15:10


## Authors


<a href="https://www.linkedin.com/in/yan-luo-96288783/">Yan Luo</a>


<a href="https://www.linkedin.com/in/nayefaboutayoun/">Nayef Abou Tayoun</a>


<!--
## Change Log
-->


<!--
| Date (YYYY-MM-DD) | Version | Changed By | Change Description      |
| ----------------- | ------- | ---------- | ----------------------- |
| 2021-06-09        | 1.0     | Yan Luo    | Tasks updates           |
| 2020-11-10        | 1.0     | Nayef      | Created the initial version |
-->


Copyright © 2021 IBM Corporation. All rights reserved.
